<a href="https://colab.research.google.com/github/JoaoVitorCoelhoG/Aprendizado-Profundo/blob/main/cm204_lab8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Instituto Tecnológico de Aeronáutica – ITA**

**Aprendizado Profundo - CM-204**

**Professores:**

Marcos Ricardo Omena de Albuquerque Maximo

`Claude Code`

---

# Laboratório 8 - Proximal Policy Optimization

**Observação**:
- **NÃO** exclua células, pois isto pode gerar problemas com o autocorretor!
- Para editar este notebook, salve uma cópia no seu Drive em ``File > Save a copy in Drive``.
- Neste laboratório, é aconselhável utilizar a GPU do Colab para o treinamento da rede. Para usar a GPU, faça o seguinte:
  - Na aba superior, selecione o menu "*Runtime*" ("Ambiente de execução");
  - Clique em "*Change runtime type*" ("Alterar tipo de ambiente de execução");
  - Selecione "GPU" na seção "*Hardware accelerator*" ("Acelerador de *hardware*") e salve.

  Verifique se a GPU está disponível executando o seguinte comando no notebook:

Neste laboratório, você implementará o **PPO** (*Proximal Policy Optimization*) para controle contínuo. A implementação é uma versão simplificada da estrutura do `ppo_continuous_action.py` do [CleanRL](https://github.com/vwxyzjn/cleanrl). Recomendo não consultar o código do `CleanRL` durante a implementação do laboratório para não perder o desafio.

Perceba que a implementação apresentada aqui é aquém em termos de desempenho do que uma implementação de uma biblioteca mais consagrada como o [Stable Baselines 3](https://stable-baselines3.readthedocs.io/en/master/) (SB3). O SB3 tem uma interface muito alto nível, que esconda praticamente tudo do algoritmo, de modo que considerei que não seria muito didático aprender o PPO através dela.

A implementação será avaliada no ambiente (*environment*) `DoubleIntervedPendulum-v5` do `Gymnasium`, um *framework* com vários ambientes para avaliação de algoritmos de aprendizado por reforço. Note que o predecessor do `Gymnasium` foi o `OpenAI Gym`, que foi desenvolvido pela OpenAI. Atualmente, o `Gymnasium` é mantido pela fundação Farama.

O `DoubleInvertedPendulum-v5` envolve equilibrar um pêndulo duplo conectado a um carrinho (*cart*) através da aplicação de força ao carrinho. A simulação física usa o Mujoco, um simulador de dinâmica de corpo rígido de alta fidelidade.

Para mais detalhes sobre os ambientes do `Gymnasium`, veja: https://gymnasium.farama.org/index.html

Para mais detalhes sobre o ambiente `DoubleInvertedPendulum-v5`, veja: https://gymnasium.farama.org/environments/mujoco/inverted_double_pendulum/

A implementação do PPO realizada aqui tem as seguintes características:
1. Redes separadas para o *actor* e para o *critic*.
2. A política adotada é Gaussiana, com média definida pela rede *actor* e o desvio padrão aprendido.
3. Coleta de amostras através de *rollout* de tamanho fixo.
4. Estimativa da Função Vantagem através de Generalized Advantage Estimation (GAE).
5. Usa-se a versão *clipped* do PPO, que é a mais popular.
6. Para melhor aproveitar o *minibatch* de dados coletado, adota-se algumas épocas de "subida de gradiente" utilizando o otimizador Adam.
7. Segue-se uma implementação básica, semelhante ao discutido no artigo original, com poucos truques adicionais, de modo a ficar didático. Implementações "sérias" do PPO adicionam vários truques que melhoram muito o desempenho do algoritmo.

## 1. Preparação do Ambiente

In [ ]:
%pip install "gymnasium[mujoco]" torch numpy matplotlib

O código a seguir realizar `import`s e define hiperparâmetros.

In [ ]:
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym


@dataclass
class Args:
    env_id: str = "InvertedDoublePendulum-v5"
    seed: int = 1
    total_timesteps: int = 150_000   # total environment steps to train for
    num_envs: int = 1                # parallel environments
    num_steps: int = 2048            # steps collected per env before each update
    learning_rate: float = 3e-4
    gamma: float = 0.99              # discount factor
    gae_lambda: float = 0.95         # GAE bias/variance trade-off
    update_epochs: int = 10          # passes over the rollout per update
    num_minibatches: int = 32        # minibatches per epoch
    clip_coef: float = 0.2           # PPO clipping epsilon
    norm_adv: bool = True            # normalize advantages per minibatch
    ent_coef: float = 0.0            # entropy bonus coefficient
    vf_coef: float = 0.5             # value loss coefficient

    # Derived sizes (filled in below)
    batch_size: int = 0
    minibatch_size: int = 0
    num_iterations: int = 0


args = Args()
args.batch_size = args.num_envs * args.num_steps
args.minibatch_size = args.batch_size // args.num_minibatches
args.num_iterations = args.total_timesteps // args.batch_size

# Reproducibility
random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print(f"batch_size={args.batch_size}  minibatch_size={args.minibatch_size}  "
      f"num_iterations={args.num_iterations}")

## 2. Ambiente do `Gymnasium`

O código a seguir define o ambiente do `Gymnasium`.

A função `make_env()` também define alguns `wrapper`s para facilitar:
- `RecordEpisodeStatistics` registra o retorno e o comprimento (em termos de *timesteps*) do episódio.
- `ClipAction` mantém as ações dentro do intervalo de ações válido.
- `NormalizeReward` reescala as recompensas durante o treinamento para estabilizar o aprendizado.

In [ ]:
def make_env(env_id, gamma):
    """
    Makes the environment and applies wrappers to it. The wrappers used are:
    - RecordEpisodeStatistics: records raw return and length of episodes
    - ClipAction: clips actions to the action space of the environment
    - NormalizeReward: normalizes rewards using a running mean and std, with discount factor gamma
    """
    env = gym.make(env_id)
    env = gym.wrappers.RecordEpisodeStatistics(env)  # records raw return/length
    env = gym.wrappers.ClipAction(env)
    env = gym.wrappers.NormalizeReward(env, gamma=gamma)
    return env

env = make_env(args.env_id, args.gamma)

assert isinstance(env.action_space, gym.spaces.Box), \
    "PPO continuous expects a Box (continuous) action space"

obs_dim = int(np.array(env.observation_space.shape).prod())
act_dim = int(np.array(env.action_space.shape).prod())
print("Observation dim:", obs_dim, "  Action dim:", act_dim)

## 3. Definição do Agente

O agente é composto por duas redes neurais:
- A rede **critic**, que estima a função valor $V(s)$ para um dado estado $s$, and
- A rede **actor**, que parametriza uma política Gaussiana policy $\pi(a|s) = \mathcal{N}(\mu(s), \sigma)$,
  em que a média $\mu(s)$ vem da rede e o logaritmo do desvio padrão é um único parâmetro independente do estado a ser aprendido. Genericamente, adotamos um desvio padrão a ser aprendido por dimensão do espaço de ações.

<font color='red'>**Você deve implementar os seguintes métodos a seguir:**</font>
- `make_critic()`: Constrói a rede *critic* como uma MLP de 3 camadas (excluindo a entrada), sendo as duas camadas escondidas com 64 neurônios. Além disso, usa-se *tanh* como função de ativação nas camadas escondidas.
- `make_actor()`: Constrói a rede *actor* como uma MLP de 3 camadas (excluindo a entrada), sendo as duas camadas escondidas com 64 neurônios. Além disso, usa-se *tanh* como função de ativação nas camadas escondidas. Note que para o caso do `InvertedDoublePendulum-v5`, em que `act_dim` vale 1, as duas redes tem o mesmo número de parâmetros, mas faça uma implementação genérica que funcione para outros valores de `act_dim`.
- `get_value()`: Retorna a estimativa da função valor $\hat{V}_{\boldsymbol{\phi}}(s_t)$ para um dado estado $s_t$. Observação: use `squeeze(-1)` na saída para remover a última dimensão do tensor, de modo a compatibilizar com o restante do código.
- `get_action_and_value()`: Calcula uma ação amostrada $a_t \sim \pi_{\boldsymbol{\theta}}(\cdot|s_t) $, o logaritmo da probabilidade (*logprob*) $\log \pi_{\boldsymbol{\theta}}(a_t|s_t)$, a entropia $S[\pi_{\boldsymbol{\theta}}](s_t)$ e o valor estimado $\hat{V}_{\boldsymbol{\phi}}(s_t)$. Use `squeeze(-1)` na saída do *critic*. Além disso, note que se você calcular a *logprob* ou a entropia para cada dimensão da Gaussiana separadamente, você ainda precisa somar ao longo da dimensão das ações (`sum(-1)`) para obter a *logprob* ou entropia total da política, respectivamente.

<details><summary><b>---Dica---</b></summary>
<p>

`Normal(mean, std)` cria uma distribuição Gaussiana com média `mean` e desvio padrão `std`. Se há mais de uma dimensão em `mean` e `std`, múltiplas Gaussianas são criadas. Ademais, o objeto criado com `Normal` tem os seguintes métodos auxiliares que são úteis aqui:
- `log_prob(x)` calcula $\log p(x)$.
- `entropy()`: calcula a entropia associada à Gaussiana em questão.

</p>
</details>

In [ ]:
from torch.distributions.normal import Normal


def init_weights(module: nn.Module, std: float = np.sqrt(2), bias_const: float = 0.0):
    """
    Initialize the weights of a linear layer using orthogonal initialization and set the bias to a constant value.
    @param module: The linear layer to initialize.
    @param std: The standard deviation for the orthogonal initialization.
    @param bias_const: The constant value to set the bias to.
    """
    if isinstance(module, nn.Linear):
        torch.nn.init.orthogonal_(module.weight, std)
        torch.nn.init.constant_(module.bias, bias_const)


class Agent(nn.Module):
    """Actor-Critic agent with separate networks for the actor and critic."""
    def __init__(self, obs_dim: int, act_dim: int):
        """
        Initialize the Actor-Critic agent with separate networks for the actor and critic.
        @param obs_dim: The dimension of the observation space.
        @param act_dim: The dimension of the action space.
        """
        super().__init__()
        self.critic = Agent.make_critic(obs_dim)
        self.actor_mean = Agent.make_actor(obs_dim, act_dim)
        init_weights(self.critic[-1], std=1.0)
        init_weights(self.actor_mean[-1], std=0.01)
        self.actor_logstd = nn.Parameter(torch.zeros(act_dim))

    @staticmethod
    def make_critic(obs_dim):
        """
        Create the critic network, which estimates the value function given the observations.
        @param obs_dim: The dimension of the observation space.
        @return: A sequential neural network representing the critic.
        """
        raise NotImplementedError()
        return nn.Sequential(
            # TODO Implement the critic network here.
        )

    @staticmethod
    def make_actor(obs_dim: int, act_dim: int):
        """
        Create the actor network, which outputs the parameters of the action distribution.
        @param obs_dim: The dimension of the observation space.
        @param act_dim: The dimension of the action space.
        @return: A sequential neural network representing the actor.
        """
        raise NotImplementedError()
        return nn.Sequential(
            # TODO Implement the actor network here.
        )

    def get_value(self, obs: torch.Tensor):
        """
        Estimate the value function for the given observations.
        @param obs: A tensor of observations.
        @return: A tensor of estimated values for each observation.
        """
        raise NotImplementedError()
        return torch.tensor(0.0)  # TODO Implement the value estimation here. Delete this line.

    def get_action_and_value(self, obs: torch.Tensor, action: torch.Tensor = None):
        """
        Sample (or evaluate) an action and return logprob, entropy, and value.
        @param obs: A tensor of observations.
        @param action: A tensor of actions (optional).
                       If action is provided, the log probability and entropy will be
                       computed for that action; otherwise, a new action will be sampled.
        @return: A tuple of the sampled action, log probability, entropy, and estimated value.
        """
        # TODO Implement the action sampling/evaluation and value estimation here.
        raise NotImplementedError()
        # probs = Normal(...)
        if action is None:
            action = probs.sample()
        # Sum log-probs across action dimensions (independent Gaussians)
        # return ...


agent = Agent(obs_dim, act_dim).to(device)
optimizer = optim.Adam(agent.parameters(), lr=args.learning_rate)
print(agent)

O código a seguir testa a implementação de `Agent`.

In [ ]:
def count_params(module: nn.Module):
    """Count the number of trainable parameters in a PyTorch module."""
    return sum(p.numel() for p in module.parameters())

np.random.seed(args.seed)
torch.random.manual_seed(args.seed)

agent = Agent(obs_dim, act_dim).to('cpu')
optimizer = optim.Adam(agent.parameters(), lr=args.learning_rate)
print(f"Number of parameters in actor: {count_params(agent.actor_mean)}")
print(f"Number of parameters in critic: {count_params(agent.critic)}")
assert count_params(agent.actor_mean) == 4865
assert count_params(agent.critic) == 4865

obs = np.array([np.zeros(obs_dim),
                np.ones(obs_dim)], dtype=np.float32)
value = agent.get_value(torch.as_tensor(obs, dtype=torch.float32))
assert value.shape == (2,)
assert value.dtype == torch.float32
assert (value - torch.tensor([-0.1670, -0.0445])).abs().max() < 1e-3

action, logprob, entropy, value = agent.get_action_and_value(torch.as_tensor(obs, dtype=torch.float32))
assert action.shape == (2, act_dim)
assert logprob.shape == (2,)
assert entropy.shape == (2,)
assert value.shape == (2,)
assert action.dtype == torch.float32
assert logprob.dtype == torch.float32
assert entropy.dtype == torch.float32
assert value.dtype == torch.float32
assert (action - torch.tensor([[-0.3383], [-0.6231]])).abs().max() < 1e-3
assert (logprob - torch.tensor([-0.9764, -1.1135])).abs().max() < 1e-3
assert (entropy - torch.tensor([1.4189, 1.4189])).abs().max() < 1e-3
assert (value - torch.tensor([-0.1670, -0.0445])).abs().max() < 1e-3

## 4. Avaliação do Agente Inicial

A seguir, cria-se um agente com pesos aleatórios e avalia-se seu desempenho no ambiente. <font color='red'>**Comente no seu relatório sobre o desempenho do agente obtido nesta etapa.**</font>

In [ ]:
agent = Agent(obs_dim, act_dim).to(device)
optimizer = optim.Adam(agent.parameters(), lr=args.learning_rate)

In [ ]:
@torch.no_grad()
def record_video(agent, env_id, name_prefix, folder="videos", n_episodes=3):
    """
    Record a video of the agent interacting with the environment for a specified number of episodes.
    @param agent: The trained agent to be evaluated.
    @param env_id: The ID of the environment to be used for evaluation.
    @param name_prefix: The prefix for the video file names.
    @param folder: The folder where the videos will be saved.
    @param n_episodes: The number of episodes to record.
    """
    env = gym.make(env_id, render_mode="rgb_array")
    env = gym.wrappers.RecordVideo(
        env, video_folder=folder, name_prefix=name_prefix,
        episode_trigger=lambda ep: True,
    )
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        while not done:
            obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            action = agent.actor_mean(obs_t).cpu().numpy()[0]
            obs, _, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
    env.close()

In [ ]:
record_video(agent, args.env_id, "random")
print("Videos saved to ./videos/")

In [ ]:
import glob
from IPython.display import Video

vids = sorted(glob.glob("videos/random-*.mp4"))
Video(vids[0], embed=True) if vids else print("No video found.")

## 5. Treinamento

Antes de realizar o treinamento, vamos primeiramente implementar funções que calculam conforme as equações apresentadas no artigo do [PPO](https://arxiv.org/pdf/1707.06347). Para isso, implemente:
- `compute_clipped_loss()`: Implementa o equivalente à função **objetivo** $L^{CLIP}(\boldsymbol{\theta})$ da versão *Clipped Surrogate Objective* do PPO. Note que no caso do artigo do PPO, considera-se um objetivo que se deseja maximizar, mas o PyTorch sempre trabalha com minimização. Assim, para que o PyTorch possa ser usado na otimização, deve-se fazer as adaptações necessárias na equação. Lembre-se que no artigo do PPO, tem-se:

\begin{equation}
L^{CLIP}(\boldsymbol{\theta}) = \mathbb{E}_t \left[ \min \left( r_t(\boldsymbol{\theta}) \hat{A}_t, \mathrm{clip}(r_t, 1-\varepsilon, 1+\varepsilon) \hat{A}_t \right) \right],
\end{equation}
em que

\begin{equation}
r_t = \dfrac{\pi_{\boldsymbol{\theta}} (a_t|s_t)}{\pi_{\boldsymbol{\theta}_{old}} (a_t|s_t)},
\end{equation}
em que $\pi_{\boldsymbol{\theta}_{old}} (a_t|s_t)$, $\varepsilon$ é um hiperparâmetro, $\hat{A}_t$ é a estimativa da função vantagem e $\mathrm{clip}(x, a, b)$ satura o valor de $x$ de acordo com os limites $a$ e $b$.

- `compute_value_loss`: Implementa a *loss* usada para treinar o *critic*. Usa-se erro médio quadrático (MSE):

\begin{equation}
L^{VF}(\boldsymbol{\phi}) = \mathbb{E}_t \left[ \frac{1}{2} \left( \hat{V}_{\boldsymbol{\phi}} - G_t \right)^2 \right],
\end{equation}
em que $\hat{V}_{\boldsymbol{\phi}}$ é a estimativa da função valor e $G_t$ é o retorno obtido experimentalmente, respectivamente.

- `compute_ppo_loss()`: Calcula a função de perda total do PPO.

<details><summary><b>---Dica---</b></summary>
<p>

- Aqui, a esperança é calculada de forma experimental, assim basta realiza uma média em cima das amostras do *minibatch*. No PyTorch, isso pode ser feito com `mean()`.
- A função `compute_ppo_loss()` assume que todas as funções de perda já estão na convenção do PyTorch (i.e., todas devem ser minimizadas).

</p>
</details>

In [ ]:
def compute_clipped_loss(advantages: torch.Tensor, ratio: torch.Tensor, clip_coef: float):
    """
    Computes the clipped loss for PPO.
    Notice that PyTorch minimizes the loss, so we return the negative of the objective.
    @param advantages: A tensor of advantages of size (batch_size, 1).
    @param ratio: A tensor of probability ratios (new policy / old policy) of size (batch_size, 1).
    @param clip_coef: The clipping coefficient for PPO.
    @return: The mean clipped loss.
    """
    raise NotImplementedError()
    # TODO Implement the clipped loss computation here.
    # return clipped_loss

def compute_value_loss(estimated_values: torch.Tensor, returns: torch.Tensor):
    """
    Computes the value loss for PPO.
    @param estimated_values: A tensor of estimated values of size (batch_size, 1).
    @param returns: A tensor of returns of size (batch_size, 1).
    @return: The mean value loss.
    """
    raise NotImplementedError()
    # TODO Implement the value loss computation here.
    # return value_loss

def compute_ppo_loss(clipped_loss: torch.Tensor, value_loss: torch.Tensor,
                     entropy_loss: torch.Tensor, vf_coef: float, ent_coef: float):
    """
    Computes the overall PPO loss.
    This method assumes that each loss component has already been computed and averaged over the batch.
    Moreover, it also assumes that each loss should be minimized.
    @param clipped_loss: The mean clipped loss.
    @param value_loss: The mean value loss.
    @param entropy_loss: The mean entropy loss.
    @param vf_coef: The coefficient for the value function loss.
    @param ent_coef: The coefficient for the entropy loss.
    @return: The overall PPO loss.
    """
    raise NotImplementedError()
    # TODO Implement the overall PPO loss computation here.
    # return ppo_loss

O código a seguir testa as implementações das funções de perda:

In [ ]:
agent = Agent(obs_dim, act_dim).to('cpu')
optimizer = optim.Adam(agent.parameters(), lr=args.learning_rate)

advantages = torch.tensor([0.0, 1.0])
obs_np = np.array([np.zeros(obs_dim),
                   np.ones(obs_dim)], dtype=np.float32)
obs = torch.as_tensor(obs_np, dtype=torch.float32)
actions = torch.tensor([2.0, 3.0])
logprobs = torch.tensor([4.0, 5.0])
_, newlogprob, entropy, newvalue = agent.get_action_and_value(
    obs, actions
)
logratio = newlogprob - logprobs
ratio = logratio.exp()
clipped_loss = compute_clipped_loss(advantages, ratio, 0.2)
assert abs(clipped_loss.item() - -7.9392e-07) < 1e-3
estimated_values = torch.tensor([0.0, 1.0, 2.0])
returns = torch.tensor([3.0, 4.0, 5.0])
value_loss = compute_value_loss(estimated_values, returns)
assert abs(value_loss.item() - 4.5) < 1e-3
ppo_loss = compute_ppo_loss(1.0, 2.0, 3.0, 0.1, 0.2)
assert abs(ppo_loss - 1.8) < 1e-3

Cada **iteração** do treinamento abaixo faz o seguinte:

1. **Rollout**: Executa `num_steps` passos no ambiente do `Gymnasium`, armazenando observações,
   ações, log-probabilidades, recompensas, flags de término (*dones*) e estimativas de valor.

2. **Generalized Advantage Estimation (GAE)**: Percorre o *rollout de trás para frente* para calcular as vantagens. O GAE é calculada da seguinte forma:

\begin{equation*}
\hat{A}_t^{\mathrm{GAE}(\gamma,\lambda)} = \sum_{l=0}^{\infty} (\gamma\lambda)^l \, \delta_{t+l},
\end{equation*}
em que $\delta_t$ é o erro TD (*Temporal-Difference*):
\begin{equation*}
\delta_t = r_t + \gamma\, V(s_{t+1}) - V(s_t).
\end{equation*}
No caso do algoritmo usado aqui, é conveniente adotar a seguinte formulação recursiva:
\begin{equation*}
\hat{A}_t = \delta_t + \gamma\lambda\, \hat{A}_{t+1}.
\end{equation*}

3. **Atualização**: Por várias épocas, amostra *minibatches* e minimiza a perda do PPO.

In [ ]:
agent = Agent(obs_dim, act_dim).to(device)
optimizer = optim.Adam(agent.parameters(), lr=args.learning_rate)

obs = torch.zeros((args.num_steps, obs_dim)).to(device)
actions = torch.zeros((args.num_steps, act_dim)).to(device)
logprobs = torch.zeros(args.num_steps).to(device)
rewards = torch.zeros(args.num_steps).to(device)
dones = torch.zeros(args.num_steps).to(device)
values = torch.zeros(args.num_steps).to(device)

global_step = 0
episode_returns = []   # for plotting later
episode_steps = []

# Reset the vector env once; thereafter it auto-resets finished episodes.
next_obs, _ = env.reset(seed=args.seed)
next_obs = torch.tensor(next_obs, dtype=torch.float32, device=device)
next_done = torch.zeros(1, device=device)

for iteration in range(1, args.num_iterations + 1):
    # ---------- 1) Collect a rollout ----------
    for step in range(args.num_steps):
        global_step += 1
        obs[step] = next_obs
        dones[step] = next_done

        # Act under the current policy WITHOUT tracking gradients
        with torch.no_grad():
            action, logprob, _, value = agent.get_action_and_value(next_obs)
            values[step] = value
        actions[step] = action
        logprobs[step] = logprob

        # ----- The Gymnasium step -----
        next_obs_np, reward, terminations, truncations, infos = env.step(action.cpu().numpy())
        next_done_np = terminations or truncations
        rewards[step] = torch.tensor(reward, dtype=torch.float32, device=device)
        next_obs = torch.tensor(next_obs_np, dtype=torch.float32, device=device)
        next_done = torch.tensor(next_done_np, dtype=torch.float32, device=device)

        # RecordEpisodeStatistics reports finished episodes in `infos`
        if "episode" in infos:
            r = float(infos["episode"]["r"])
            episode_returns.append(r)
            episode_steps.append(global_step)

        if next_done:
            next_obs_np, _ = env.reset()
        next_obs = torch.tensor(next_obs_np, dtype=torch.float32, device=device)

    # ---------- 2) Compute advantages with GAE ----------
    with torch.no_grad():
        next_value = agent.get_value(next_obs)#.reshape(1, -1)
        advantages = torch.zeros_like(rewards).to(device)
        lastgaelam = 0
        for t in reversed(range(args.num_steps)):
            if t == args.num_steps - 1:
                nextnonterminal = 1.0 - next_done
                nextvalues = next_value
            else:
                nextnonterminal = 1.0 - dones[t + 1]
                nextvalues = values[t + 1]
            delta = rewards[t] + args.gamma * nextvalues * nextnonterminal - values[t]
            advantages[t] = lastgaelam = delta + args.gamma * args.gae_lambda * nextnonterminal * lastgaelam
        returns = advantages + values

    # ---------- 3) PPO update ----------
    b_obs = obs
    b_logprobs = logprobs
    b_actions = actions
    b_advantages = advantages
    b_returns = returns
    b_values = values

    b_inds = np.arange(args.batch_size)
    clipfracs = []
    for epoch in range(args.update_epochs):
        np.random.shuffle(b_inds)
        for start in range(0, args.batch_size, args.minibatch_size):
            mb_inds = b_inds[start:start + args.minibatch_size]

            _, newlogprob, entropy, newvalue = agent.get_action_and_value(
                b_obs[mb_inds], b_actions[mb_inds]
            )
            logratio = newlogprob - b_logprobs[mb_inds]
            ratio = logratio.exp()

            with torch.no_grad():
                # Approximate KL (Schulman) and clip fraction, for monitoring
                approx_kl = ((ratio - 1) - logratio).mean()
                clipfracs.append(((ratio - 1.0).abs() > args.clip_coef).float().mean().item())

            mb_advantages = b_advantages[mb_inds]
            if args.norm_adv:
                mb_advantages = (mb_advantages - mb_advantages.mean()) / (mb_advantages.std() + 1e-8)

            clipped_loss = compute_clipped_loss(mb_advantages, ratio, args.clip_coef)

            value_loss = compute_value_loss(newvalue, b_returns[mb_inds])

            entropy_loss = -entropy.mean()
            loss = compute_ppo_loss(clipped_loss, value_loss, entropy_loss, args.vf_coef, args.ent_coef)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Explained variance: how well the value function predicts returns (1.0 is perfect)
    y_pred, y_true = b_values.cpu().numpy(), b_returns.cpu().numpy()
    var_y = np.var(y_true)
    explained_var = np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y

    recent_return = np.mean(episode_returns[-10:]) if episode_returns else float("nan")
    print(
        f"iter {iteration:3d}/{args.num_iterations} | step {global_step:7d} | "
        f"return(avg10) {recent_return:7.1f} | "
        f"approx_kl {approx_kl.item():.4f} | clipfrac {np.mean(clipfracs):.3f} | "
        f"expl_var {explained_var:5.2f}"
    )

env.close()
print("Training done.")

## 6. Avaliação da Política

A célula a seguir mostra a curva de aprendizado. Como o aprendizado é ruidoso, adota-se também uma média móvel com janela de 10 episódios (curva laranja).

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(episode_steps, episode_returns, alpha=0.4, label="Episode Return")
if len(episode_returns) >= 10:
    kernel = np.ones(10) / 10
    smoothed = np.convolve(episode_returns, kernel, mode="valid")
    plt.plot(episode_steps[9:], smoothed, label="Moving Average (10)")
plt.xlabel("Environment Steps")
plt.ylabel("Episode Return")
plt.title("PPO on InvertedDoublePendulum-v5")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

A célula a seguir desativa o aprendizado e avalia a política 20 vezes.

In [ ]:
@torch.no_grad()
def evaluate(agent, env_id, n_episodes=20, deterministic=False):
    env = gym.make(env_id)
    returns = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        total = 0.0
        while not done:
            obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            if deterministic:
                action = agent.actor_mean(obs_t)            # use the mean action
            else:
                action, _, _, _ = agent.get_action_and_value(obs_t)
            action_np = action.cpu().numpy()[0]
            obs, reward, terminated, truncated, _ = env.step(action_np)
            total += reward
            done = terminated or truncated
        returns.append(total)
    env.close()
    return np.mean(returns), np.std(returns)


mean_r, std_r = evaluate(agent, args.env_id, n_episodes=20)
print(f"Mean evaluation return: {mean_r:.1f} +/- {std_r:.1f}")

A seguir, grava-se e mostra-se vídeos do agente treinado.

In [ ]:
record_video(agent, args.env_id, "trained")
print("Videos saved to ./videos/")

In [ ]:
import glob
from IPython.display import Video

vids = sorted(glob.glob("videos/trained-*.mp4"))
Video(vids[0], embed=True) if vids else print("No video found.")

# 7. Entrega

A entrega consiste do notebook no formato **.ipynb** e de um relatório, submetida através do Google Classroom. Modificações nos arquivos do código base são permitidas, desde que o nome e a interface dos scripts “main” não sejam alterados. A princípio, não há limitação de número de páginas para o relatório, mas pede-se que seja sucinto. O relatório deve conter:
- Figuras que comprovem o funcionamento do seu código.
- Demais solicitações feitas ao longo do roteiro.

Por limitações do Google Classroom (e por motivo de facilitar a automatização da correção), entregue seu laboratório com todos os arquivos num único arquivo **.zip** (**não** utilize outras tecnologias de compactação de arquivos) com o seguinte padrão de nome: **“<login_email_google_education>_labX.zip”**. Por exemplo, no meu caso, meu login Google Education é **marcos.maximo**, logo eu entregaria o lab 1 como **“marcos.maximo_lab1.zip”**. **Não** crie subpastas para os arquivos da sua entrega, **deixe todos os arquivos na “raiz” do .zip**.